# Well Coordinate Assignment & Well-Level Production/Disposition Estimates

**Pipeline stage:** Step 2 of the Texas RRC oil production data pipeline (follows the primary
extraction notebook that produces `texas_prod_disp.parquet` and `well_api_lease.parquet`).

This notebook:

1. Assigns geographic coordinates (latitude/longitude) to individual Texas oil wells, using RRC's
   well-location shapefiles.
2. Builds a per-well approximation of monthly oil/casinghead-gas disposition, by taking the
   lease-level disposition data from Step 1 and splitting it equally across every well known to sit
   on that lease.

Every well keeps its own individual coordinate — leases are never collapsed down to a single
averaged point. Instead, each lease's monthly production/disposition volumes are divided equally
across all wells with known coordinates on that lease, with the well count kept as an explicit
column (`n_wells_with_coordinates`) so the approximation is fully transparent and auditable.

Because this notebook processes the full, multi-year production history and performs a
row-multiplying join (see Section 0 below), it can require more memory than a single kernel session
comfortably holds. Section 5 processes the merge **chunk by chunk (one calendar year at a time)**,
checkpointing each chunk to disk, so that a kernel crash only costs you the current chunk — re-running
the cell picks up where it left off instead of starting over.

## 0. Overview: Why We Need Two Join Keys, and How Everything Fits Together

Before writing any code, it's worth understanding *why* this notebook is structured the way it is,
since almost every step exists to solve one of two problems: **(a)** two files describe the same
lease with slightly different raw fields, or **(b)** wells and their coordinates live in a totally
separate data source (shapefiles) that has to be matched in by a different identifier.

### The two input files (both produced by Step 1 of this pipeline)

| File | Granularity | Key raw fields | Problem |
|---|---|---|---|
| `well_api_lease.parquet` | one row per **well** | `oil_gas_code`, `district_no`, `lease_no` (which lease it's on), plus `api_county_code` / `api_unique_no` (its API number) | Has no coordinates |
| `texas_prod_disp.parquet` | one row per **lease per month** | `oil_gas_code`, `district_no`, `lease_no` | Has no well identity or coordinates at all — it's lease-level only |

Neither file has coordinates. Coordinates only exist in a **third**, independent data source: RRC's
well-location **shapefiles**, which identify wells by **API number**, not by lease number.

### Two stable keys solve this

- **`lease_key`** — built by normalizing and concatenating `oil_gas_code + "_" + district_no + "_" +
  lease_no` (zero-padding district/lease numbers, upper-casing codes). This is the join key between
  `well_api_lease.parquet` and `texas_prod_disp.parquet`. It has to be *constructed* rather than used
  raw because the same lease can be written with inconsistent zero-padding or casing across files
  (e.g. `"277"` vs `"00277"`), so a raw string join would silently drop matches.
- **`api8`** — the RRC's 8-digit API well identifier (3-digit county code + 5-digit unique well
  number). This is the join key between `well_api_lease.parquet` and the well shapefiles. It also has
  to be constructed/normalized because API numbers show up in the shapefiles in several different raw
  formats (8-digit, 10-digit with a leading state code, 12-digit `APINUM`, etc.) that all need to
  collapse to the same 8-digit form to match.

### The full join chain, end to end

```
well_api_lease.parquet  ──(build lease_key + api8)──▶  lease_wells
                                                            │
well shapefiles (255 county zips) ──(parse, keep points,   │  merge on api8
  extract api8 from API/API10/APINUM columns)──▶ well_coords_one
                                                            │
                                                            ▼
                                          lease_well_coordinates
                                   (one row per WELL: lease_key, api8, lon/lat)
                                                            │
texas_prod_disp.parquet ──(build lease_key, filter to    │  merge on lease_key
  post-2012 dates)──▶ prod_disp                            │
                                                            ▼
                                    one row per WELL per MONTH
                        (lease-level $ duplicated across every well on the lease —
                         see the row-explosion note below)
                                                            │
                divide production/disposition columns by n_wells_with_coordinates
                                                            │
                                                            ▼
                                    prod_per_well_approx  (final output)
```

### The "row explosion" and why we divide instead of collapsing to a centroid

When we merge lease-level production onto well-level coordinates on `lease_key`, a lease with 5 wells
turns **1 production row into 5 identical rows** — one per well, each still showing the *full lease
volume*. Left as-is, that's wrong: it looks like each of the 5 wells independently produced the full
lease total.

There are two conceptually different ways to fix this:

1. **Collapse to a centroid:** compute one averaged coordinate per lease and join production to that
   single point. This avoids overcounting, but throws away well-level spatial resolution — you get
   one point per lease, not one per well.
2. **Divide instead of collapse** (the approach used in this notebook): keep every well as its own
   row, but divide every production/disposition column by the number of wells with known coordinates
   on that lease (`n_wells_with_coordinates`). Every well now shows its *equal share* of the lease's
   volume, and downstream spatial analysis (e.g. mapping flaring by well location) gets real per-well
   points instead of one averaged point per lease.

This notebook uses approach **2**. `n_wells_with_coordinates` is kept as an explicit output column
specifically so that anyone using this data can see exactly how many ways each lease's volume was
split — it is not a hidden assumption.

> ⚠️ **Important caveat:** this is an **equal-split approximation**, not measured well-level
> production. Individual wells on a lease may produce very different volumes in reality (a stronger
> well vs. a stripper well on the same lease, for example). Treat these outputs as spatial proxies
> for approximate location-weighted analysis, not as precise per-well measurements.

## 1. How to Download the Well Shapefiles ("Well Layers by County")

The production/disposition data (Step 1 of this pipeline) came from RRC's Production Data Query
dump. Well **coordinates** come from a completely separate RRC product: per-county well-location
shapefiles, referred to on RRC's site as **"Well Layers by County"** (sometimes bundled together with
survey and pipeline layers as **"All Layers by County"**).

### Step-by-step download instructions

1. Go to the RRC data download page: https://www.rrc.texas.gov/resource-center/research/data-sets-available-for-download/
2. Find the **"Well"** section and click the **"ArcView Shape File"** link. This takes you to RRC's
   Managed File Transfer (MFT) portal — a file browser listing one zip file per Texas county (plus a
   couple of extra/statewide entries), for a total of roughly **255 zip files**.
   - If you already know a specific county's number, the [RRC Public GIS Viewer](https://gis.rrc.texas.gov/GISViewer/)
     is a handy way to look it up and preview well locations before downloading.
3. On the MFT portal page, select the county zip files you want. To reproduce this pipeline exactly,
   select **all** counties (there's a "select all" option) — the notebook expects the full statewide
   set of ~255 zips.
4. Download the selected files. **Do not unzip them** — like the production data in Step 1, this
   notebook reads directly out of the zip files in-memory via `zipfile` / `geopandas`.
5. Place all downloaded county zip files into a single folder:

   ```
   <project_root>/data/raw/texas/Wells/
   ```

   This is the `shapefile_zip_folder` referenced in the config cell below.

### What's inside each county zip

Each county's zip archive contains that county's data as several **discrete shapefiles** (RRC does
not lump layers together):

- **Surface well locations** — where each well originates at the surface. For a vertical well, this
  is effectively "the" well location.
- **Bottom well locations** — where each well terminates downhole. For vertical wells this is
  identical to the surface location; for horizontal/directional wells it can differ substantially
  (and a single well can have multiple bottom points).
- **Well arcs/lines** — connecting surface to bottom, for directional wells (not used in this
  notebook — we only need point geometries).

### Projection & format details (needed later when reading the files)

| Property | Value |
|---|---|
| File format | ArcView Shapefile (`.shp` + companion `.dbf`/`.shx`/`.prj`) |
| Projection | Geographic |
| Units | Decimal degrees |
| Datum | **NAD27** (not WGS84 — must be reprojected, which the notebook does below) |

### Update cadence

RRC refreshes this dataset roughly **twice a week**. Re-download periodically if you need the most
current well locations. Note that unlike the production dump (Step 1), this is a live snapshot, not a
historical time series — there's no "as of" date attached to individual well points beyond "most
recent RRC records.

## 2. Setup (Imports & Logging)

Run this cell first, and re-run it after any kernel crash. `geopandas` handles all spatial
operations (reading shapefiles, reprojecting, computing point geometry); `tqdm` gives a progress bar
while looping over the ~255 shapefile zips; `gc` is used throughout the chunked section (Section 5)
to keep peak memory down.

In [11]:
# ── Imports + logging ─────────────────────────────────────────────────────────
# Run this first after any kernel crash.

from pathlib import Path
import re
import zipfile
import warnings
import gc
import logging
import sys

import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger(__name__)

log.info("✓ Imports + logging ready.")


09:45:11 [INFO] ✓ Imports + logging ready.


## 3. Configuration

- `well_api_lease_path` — output of Step 1, Part 2 (`well_api_lease.parquet`): one row per well, with
  its lease number and API components.
- `production_disp_path` — output of Step 1, Part 1 (`texas_prod_disp.parquet`): monthly lease-level
  disposition data.
- `shapefile_zip_folder` — the folder you downloaded the ~255 county zip files into (Section 1).
- `output_folder` — where this notebook's outputs are written; same shared cleaned-data folder used
  throughout the pipeline.
- `checkpoint_folder` — where per-year intermediate results from the chunked processing step
  (Section 5) are cached. If this notebook crashes partway through, re-running Section 5 will skip
  any year whose checkpoint file already exists instead of redoing the work.
- `POST_DATE_FILTER` — restricts the well-level approximation to more recent data (post-2012), where VIIRS data is available.

In [12]:
# ── Config ────────────────────────────────────────────────────────────────────

well_api_lease_path   = Path("../../../../data/processed/texas/well_api_lease.parquet")
production_disp_path  = Path("../../../../data/processed/texas/texas_prod_disp.parquet")
shapefile_zip_folder  = Path("../../../../data/raw/texas/Wells")

output_folder = Path("../../../../data/processed/texas")
output_folder.mkdir(parents=True, exist_ok=True)

checkpoint_folder = output_folder / "prod_per_well_approx_checkpoints"
checkpoint_folder.mkdir(parents=True, exist_ok=True)

POST_DATE_FILTER = "2012-01-01"   # only dates after this are used for the well-level approximation since VIIRS is available only after 2012

log.info("✓ Config set. Checkpoints will be written to: %s", checkpoint_folder)


09:45:11 [INFO] ✓ Config set. Checkpoints will be written to: ../../../../data/processed/texas/prod_per_well_approx_checkpoints


## 4. Part A — Assigning Coordinates to Individual Wells

### 4.1 Load the Step 1 Outputs

Loads the well/lease/API mapping and the lease-level disposition data produced by the previous
notebook in the pipeline. Column names are lowercased for `well_api_lease.parquet` — it was saved
with uppercase raw RRC column names in Step 1; the disposition file was already lowercased in Step 1,
but we lowercase it again here too, defensively, in case that ever changes upstream.

In [13]:
lease_wells = pd.read_parquet(well_api_lease_path)
prod_disp = pd.read_parquet(production_disp_path)

print("Lease/well shape:", lease_wells.shape)
print("Production disposition shape:", prod_disp.shape)

print("\nLease/well columns:")
print(lease_wells.columns.tolist())

print("\nProduction disposition columns:")
print(prod_disp.columns.tolist())


Lease/well shape: (587614, 9)
Production disposition shape: (16761949, 29)

Lease/well columns:
['OIL_GAS_CODE', 'DISTRICT_NO', 'LEASE_NO', 'WELL_NO', 'API_COUNTY_CODE', 'API_UNIQUE_NO', 'COUNTY_NAME', 'WELLBORE_LOCATION_CODE', 'API_NO']

Production disposition columns:
['oil_gas_code', 'district_no', 'lease_no', 'field_no', 'operator_no', 'operator_name', 'oil_pipeline_bbl', 'oil_truck_bbl', 'oil_tankcar_bbl', 'oil_tank_cleaning_bbl', 'oil_circulating_bbl', 'oil_lost_stolen_bbl', 'oil_bsw_repressure_bbl', 'oil_legacy_bbl', 'oil_skimmed_bbl', 'oil_scrubber_bbl', 'oil_no_disp_code_bbl', 'csgd_field_ops_fuel_mcf', 'csgd_transmission_mcf', 'csgd_processing_plant_mcf', 'csgd_vented_flared_mcf', 'csgd_gas_lift_mcf', 'csgd_repressure_mcf', 'csgd_carbon_black_mcf', 'csgd_underground_storage_mcf', 'csgd_no_disp_code_mcf', 'oil_sold_total_bbl', 'total_vented_flared_mcf', 'date']


In [14]:
lease_wells.columns = lease_wells.columns.str.lower()
prod_disp.columns = prod_disp.columns.str.lower()


### 4.2 Build the Stable `lease_key`

As explained in Section 0, `lease_key` is the stable identifier used to join the well/lease file and
the production file. It's built by normalizing each raw field (strip whitespace, upper-case, drop
trailing `.0` artifacts from any float-like parsing), zero-padding `district_no` and `lease_no` where
they're purely numeric (RRC districts like `"8A"` or `"7B"` are left alone), then concatenating
`oil_gas_code + "_" + district_no + "_" + lease_no`.

This is applied to `lease_wells` here in Part A. It is applied to `prod_disp` later, in Part B
(Section 5), right before it's needed — but using these exact same functions, so the two files
produce identically-formatted keys.

In [15]:
def clean_text_series(s):
    return (
        s.astype("string")
         .str.strip()
         .str.upper()
         .str.replace(r"\.0$", "", regex=True)
    )

def normalize_district(s):
    s = clean_text_series(s)

    # RRC districts can be 01, 02, 03, 04, 05, 06, 7B, 7C, 08, 8A, 09, 10, etc.
    # If purely numeric, pad to 2 digits. Leave 8A, 7B, etc. as-is.
    return s.apply(lambda x: x.zfill(2) if pd.notna(x) and x.isdigit() else x)

def normalize_lease_no(s):
    s = clean_text_series(s)

    # Many RRC lease numbers are 5 digits.
    # If your source uses a different convention, inspect before changing.
    return s.apply(lambda x: x.zfill(5) if pd.notna(x) and x.isdigit() else x)

def normalize_oil_gas_code(s):
    return clean_text_series(s)

def add_lease_key(df):
    df["oil_gas_code_norm"] = normalize_oil_gas_code(df["oil_gas_code"])
    df["district_no_norm"] = normalize_district(df["district_no"])
    df["lease_no_norm"] = normalize_lease_no(df["lease_no"])

    df["lease_key"] = (
        df["oil_gas_code_norm"] + "_" +
        df["district_no_norm"] + "_" +
        df["lease_no_norm"]
    )
    return df

lease_wells = add_lease_key(lease_wells)

lease_wells[["oil_gas_code", "district_no", "lease_no", "lease_key"]].head()


,oil_gas_code,district_no,lease_no,lease_key
0,O,08,00277,O_08_00277
1,O,08,00277,O_08_00277
2,O,08,00287,O_08_00287
3,O,08,00291,O_08_00291
4,O,08,00291,O_08_00291


### 4.3 Build the Normalized `api8`

The Texas RRC shapefiles use an 8-digit API number (usually API numbers are 10 digits, but Texas
strips the leading `"42"` state code). The 8-digit code is 3 digits for county + 5 digits for the
well within that county. Note that a lease number is only unique *within* a district, and a district
spans many counties — so `api8` and `lease_key` are two genuinely different identifiers, not
interchangeable.

`well_api_lease.parquet` gives us the raw components (`api_county_code`, `api_unique_no`) as well as
a combined `api_no` field. We build `api8` from the components first, and fall back to normalizing
`api_no` if the components are missing/invalid.

In [16]:
def clean_digits(value):
    """
    Clean a scalar value and return only digits as a string.
    """
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    text = re.sub(r"\.0$", "", text)
    text = re.sub(r"\D", "", text)

    if text == "":
        return pd.NA

    return text


def clean_digits_series(s):
    return (
        s.astype("string")
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
         .str.replace(r"\D", "", regex=True)
    )


def normalize_rrc_api8(value):
    """
    Normalize an API-like value to RRC GIS API format:

        API8 = county FIPS/API county code 3 digits + unique API number 5 digits

    Handles:
    - 8-digit RRC API: CCCNNNNN
    - 10-digit Texas API with state code: 42CCCNNNNN
    - 10-digit RRC API10: CCCNNNNNSS, where SS is sidetrack code
    - 12-digit APINUM: 42CCCNNNNNSS
    """
    digits = clean_digits(value)

    if pd.isna(digits):
        return pd.NA

    # RRC GIS API field: county3 + unique5
    if len(digits) == 8:
        return digits

    # Could be either:
    # - 42 + API8
    # - API8 + sidetrack2
    if len(digits) == 10:
        if digits.startswith("42"):
            return digits[2:10]
        else:
            return digits[:8]

    # RRC APINUM: 42 + API8 + sidetrack2
    if len(digits) >= 12 and digits.startswith("42"):
        return digits[2:10]

    return pd.NA

log.info("✓ API8 normalization functions defined.")


09:45:13 [INFO] ✓ API8 normalization functions defined.


In [17]:
county = clean_digits_series(lease_wells["api_county_code"]).str.zfill(3)
unique = clean_digits_series(lease_wells["api_unique_no"]).str.zfill(5)

lease_wells["api8_from_components"] = county + unique

# Invalid rows become <NA>
lease_wells.loc[
    lease_wells["api8_from_components"].isna() |
    (lease_wells["api8_from_components"].str.len() != 8),
    "api8_from_components"
] = pd.NA

lease_wells["api8_from_api_no"] = lease_wells["api_no"].apply(normalize_rrc_api8)
lease_wells["api8"] = lease_wells["api8_from_components"]

lease_wells.loc[
    lease_wells["api8"].isna(),
    "api8"
] = lease_wells["api8_from_api_no"]

valid_api8_rate = lease_wells["api8"].notna().mean()
print(f"Lease/well rows with valid API8: {valid_api8_rate:.2%}")

lease_wells[
    [
        "api_county_code",
        "api_unique_no",
        "api_no",
        "api8_from_components",
        "api8_from_api_no",
        "api8"
    ]
].head(10)


Lease/well rows with valid API8: 100.00%


,api_county_code,api_unique_no,api_no,api8_from_components,api8_from_api_no,api8
0,049,31721,04931721,04931721,04931721,04931721
1,049,32950,04932950,04932950,04932950,04932950
2,049,05559,04905559,04905559,04905559,04905559
3,049,05919,04905919,04905919,04905919,04905919
4,049,80560,04980560,04980560,04980560,04980560
5,049,30486,04930486,04930486,04930486,04930486
6,049,04704,04904704,04904704,04904704,04904704
7,049,35384,04935384,04935384,04935384,04935384
8,049,05369,04905369,04905369,04905369,04905369
9,049,80563,04980563,04980563,04980563,04980563


### 4.4 Read the Well Shapefiles

Loops over every county zip in `shapefile_zip_folder` (downloaded in Section 1), opens each one, and
reads every `.shp` layer inside it directly from the zip (no manual extraction needed — geopandas
supports `zip://path!file.shp` URIs). Column names are upper-cased for consistency with the rest of
this notebook's naming, **except** the geometry column, which geopandas always expects to find
lower-cased as `geometry`. Source zip/shapefile names are kept as metadata columns so any parsing
issues can be traced back to their origin file. Per RRC's documentation (Section 1), the coordinate
reference system is NAD27 if not already specified in the shapefile; every layer is reprojected to
WGS84 (`EPSG:4326`) immediately after reading so everything downstream uses standard lon/lat.

In [18]:
zip_files = sorted(shapefile_zip_folder.glob("*.zip"))

print(f"Found {len(zip_files)} zip files")
print(zip_files[:5])


Found 255 zip files
[PosixPath('../../../../data/raw/texas/Wells/well001.zip'), PosixPath('../../../../data/raw/texas/Wells/well003.zip'), PosixPath('../../../../data/raw/texas/Wells/well005.zip'), PosixPath('../../../../data/raw/texas/Wells/well007.zip'), PosixPath('../../../../data/raw/texas/Wells/well009.zip')]


In [19]:
def uppercase_non_geometry_columns(gdf):
    """
    Uppercase all non-geometry columns while preserving the active geometry column.
    """
    geom_col = gdf.geometry.name

    rename_map = {
        col: col.upper()
        for col in gdf.columns
        if col != geom_col
    }

    gdf = gdf.rename(columns=rename_map)

    # Make sure the active geometry column is still set
    gdf = gdf.set_geometry(geom_col)

    return gdf


In [20]:
gdfs = []
errors = []

for zip_path in tqdm(zip_files):
    try:
        with zipfile.ZipFile(zip_path) as z:
            shp_names = [
                name for name in z.namelist()
                if name.lower().endswith(".shp")
            ]

        if len(shp_names) == 0:
            errors.append((zip_path.name, "No .shp file found inside zip"))
            continue

        for shp_name in shp_names:
            zip_uri = f"zip://{zip_path.as_posix()}!{shp_name}"

            try:
                gdf = gpd.read_file(zip_uri, engine="pyogrio")
            except Exception:
                gdf = gpd.read_file(zip_uri)

            if gdf.empty:
                continue

            # Preserve geometry column correctly
            gdf = uppercase_non_geometry_columns(gdf)

            # Add source metadata
            gdf["SOURCE_ZIP"] = zip_path.name
            gdf["SOURCE_SHP"] = shp_name

            # RRC manual says:
            # Projection: Geographic
            # Units: Decimal Degrees
            # Datum: NAD27
            #
            # If CRS is missing, assign NAD27.
            if gdf.crs is None:
                gdf = gdf.set_crs("EPSG:4267")

            # Convert to WGS84 lon/lat
            gdf = gdf.to_crs("EPSG:4326")

            gdfs.append(gdf)

    except Exception as e:
        errors.append((zip_path.name, str(e)))

print(f"Read {len(gdfs)} shapefile layers")
print(f"Errors: {len(errors)}")
errors[:10]


  0%|          | 0/255 [00:00<?, ?it/s]

Read 733 shapefile layers
Errors: 0


[]

In [21]:
if len(gdfs) == 0:
    raise RuntimeError(
        "No shapefile layers were read. Check the errors list, folder path, and zip contents."
    )

wells_geo_all = pd.concat(gdfs, ignore_index=True)

# Convert back to GeoDataFrame, preserving whatever the geometry column is called
wells_geo_all = gpd.GeoDataFrame(
    wells_geo_all,
    geometry=gdfs[0].geometry.name,
    crs="EPSG:4326"
)

print("Combined shape:", wells_geo_all.shape)
print("CRS:", wells_geo_all.crs)
print("Geometry column:", wells_geo_all.geometry.name)

print("\nGeometry types:")
print(wells_geo_all.geometry.geom_type.value_counts(dropna=False))

print("\nColumns:")
print(wells_geo_all.columns.tolist())


Combined shape: (2958487, 20)
CRS: EPSG:4326
Geometry column: geometry

Geometry types:
Point              2772561
LineString          185748
MultiLineString        178
Name: count, dtype: int64

Columns:
['BOTTOM_ID', 'SURFACE_ID', 'SYMNUM', 'APINUM', 'RELIAB', 'API10', 'API', 'LONG27', 'LAT27', 'LONG83', 'LAT83', 'OUT_FIPS', 'CWELLNUM', 'RADIOACT', 'WELLID', 'STCODE', 'geometry', 'SOURCE_ZIP', 'SOURCE_SHP', 'SHAPE_LEN']


### 4.5 Keep Only Point Geometry

The well zip files contain surface well data as point geometry, bottom well data as point geometry,
and surface-to-bottom paths as more complex line geometry (for directional wells). We only need
point locations, so everything else is dropped. `MultiPoint` geometries (a handful of edge cases)
are collapsed to their first point so every row has a single, simple `Point`.

In [22]:
wells_geo = wells_geo_all[
    wells_geo_all.geometry.notna() &
    wells_geo_all.geometry.geom_type.isin(["Point", "MultiPoint"])
].copy()

print("Point well records:", wells_geo.shape)
print(wells_geo.geometry.geom_type.value_counts())


Point well records: (2772561, 20)
Point    2772561
Name: count, dtype: int64


In [23]:
wells_geo["geometry"] = wells_geo.geometry.apply(
    lambda geom: geom.geoms[0] if geom.geom_type == "MultiPoint" else geom
)

wells_geo = gpd.GeoDataFrame(wells_geo, geometry="geometry", crs="EPSG:4326")


### 4.6 Extract `api8` From the Shapefile's Own API Columns

Shapefiles store the API number under one of several possible column names, and in one of three
lengths: 8-digit `API`, 10-digit `API10`, or 12-digit `APINUM`. `coalesce_api8_from_row` checks each
candidate column in preference order and normalizes whichever one is present using the same
`normalize_rrc_api8` function defined in Section 4.3, so shapefile API numbers and
`well_api_lease.parquet` API numbers end up in an identical, directly comparable format.

In [24]:
def coalesce_api8_from_row(row):
    """
    Prefer API, then API10, then APINUM, then other possible API-ish names.
    Return normalized RRC API8.
    """
    candidate_cols = [
        "API",
        "API10",
        "APINUM",
        "API_NO",
        "APINO",
        "API_NUM",
        "API_NUMBER",
        "UWI"
    ]

    for col in candidate_cols:
        if col in row.index:
            value = normalize_rrc_api8(row[col])
            if pd.notna(value):
                return value

    return pd.NA


In [25]:
wells_geo["api8"] = wells_geo.apply(coalesce_api8_from_row, axis=1)

print("Well GIS rows with API8:", wells_geo["api8"].notna().mean())
wells_geo[["api8"] + [c for c in ["API", "API10", "APINUM", "SOURCE_ZIP", "SOURCE_SHP"] if c in wells_geo.columns]].head()


Well GIS rows with API8: 0.7380663581432474


,api8,API,API10,APINUM,SOURCE_ZIP,SOURCE_SHP
0,00132761,00132761,00132761,4200132761,well001.zip,well001b.shp
1,00101708,00101708,00101708,4200101708,well001.zip,well001b.shp
2,00100126,00100126,00100126,4200100126,well001.zip,well001b.shp
3,00131523,00131523,00131523,4200131523,well001.zip,well001b.shp
4,00132335,00132335,00132335,4200132335,well001.zip,well001b.shp


In [26]:
wells_geo = wells_geo.dropna(subset=["api8"]).copy()

wells_geo["longitude"] = wells_geo.geometry.x
wells_geo["latitude"] = wells_geo.geometry.y

wells_geo[["longitude", "latitude"]].describe()


,longitude,latitude
count,2.046334e+06,2.046334e+06
mean,-9.937510e+01,3.163559e+01
std,2.519270e+00,2.067646e+00
min,-1.065368e+02,2.585656e+01
25%,-1.015927e+02,3.035809e+01
50%,-9.920437e+01,3.200190e+01
75%,-9.764186e+01,3.294880e+01
max,-9.352909e+01,3.649904e+01


### 4.7 Sanity-Check Coordinates Against Texas's Boundaries

A cheap guard against bad shapefile records: any point outside a generous bounding box around Texas
is almost certainly a data error (wrong CRS, corrupted coordinate, etc.) rather than a real well.

In [27]:
suspicious_coords = wells_geo[
    ~wells_geo["longitude"].between(-107, -93) |
    ~wells_geo["latitude"].between(25, 37)
]

print("Suspicious coordinate rows:", len(suspicious_coords))
suspicious_coords[["api8", "longitude", "latitude", "SOURCE_ZIP", "SOURCE_SHP"]].head()


Suspicious coordinate rows: 0


,api8,longitude,latitude,SOURCE_ZIP,SOURCE_SHP


### 4.8 Prefer Surface Locations Over Bottom Locations, and Deduplicate

A single well can legitimately appear more than once across the shapefiles — for instance, once in
the surface-well layer and once in the bottom-well layer, or across county boundaries. Since we only
want one coordinate per well, we:

1. Label each row `surface`, `bottom`, or `unknown` by pattern-matching the source shapefile's
   filename (RRC's naming convention ends surface-well shapefiles in `S.shp` and bottom-well
   shapefiles in `B.shp`).
2. Keep `surface` and `unknown` layers (drop `bottom`-only records where a surface record also
   exists), since a well's surface location is the more meaningful "where is this well" point for our
   purposes.
3. When duplicates remain for the same `api8`, prefer `surface` over `unknown` over `bottom`, and
   (where available) prefer higher `RELIAB` (RRC's own reliability code for the coordinate), keeping
   exactly one row per `api8`.

In [28]:
wells_geo["source_shp_lower"] = wells_geo["SOURCE_SHP"].astype(str).str.lower()

wells_geo["well_layer_type"] = "unknown"

wells_geo.loc[
    wells_geo["source_shp_lower"].str.contains(r"s\.shp$", regex=True),
    "well_layer_type"
] = "surface"

wells_geo.loc[
    wells_geo["source_shp_lower"].str.contains(r"b\.shp$", regex=True),
    "well_layer_type"
] = "bottom"

wells_geo["well_layer_type"].value_counts(dropna=False)


well_layer_type
bottom     1031641
surface    1014693
Name: count, dtype: int64

In [29]:
surface_wells_geo = wells_geo[wells_geo["well_layer_type"].isin(["surface", "unknown"])].copy()

print("Surface/unknown well records:", surface_wells_geo.shape)


Surface/unknown well records: (1014693, 25)


In [30]:
surface_wells_geo["layer_priority"] = surface_wells_geo["well_layer_type"].map({
    "surface": 1,
    "unknown": 2,
    "bottom": 3
}).fillna(9)

sort_cols = ["api8", "layer_priority"]

if "RELIAB" in surface_wells_geo.columns:
    sort_cols.append("RELIAB")

well_coords_one = (
    surface_wells_geo
    .sort_values(sort_cols)
    .drop_duplicates(subset=["api8"], keep="first")
    .copy()
)

keep_cols = [
    "api8",
    "longitude",
    "latitude",
    "geometry",
    "SOURCE_ZIP",
    "SOURCE_SHP",
    "well_layer_type"
]

for optional_col in ["API", "API10", "APINUM", "LAT27", "LONG27", "LAT83", "LONG83", "RELIAB", "SYMBOL", "SYMNUN", "SYMNUM", "WELLID"]:
    if optional_col in well_coords_one.columns and optional_col not in keep_cols:
        keep_cols.append(optional_col)

well_coords_one = well_coords_one[keep_cols].copy()
well_coords_one = gpd.GeoDataFrame(well_coords_one, geometry="geometry", crs="EPSG:4326")

print("Unique API8 coordinate records:", well_coords_one.shape)
well_coords_one.head()


Unique API8 coordinate records: (1014636, 17)


,api8,longitude,latitude,geometry,SOURCE_ZIP,SOURCE_SHP,well_layer_type,API,API10,APINUM,LAT27,LONG27,LAT83,LONG83,RELIAB,SYMNUM,WELLID
5664,00100001,-96.032325,32.001892,POINT (-96.03232 32.00189),well001.zip,well001s.shp,surface,00100001,NaN,NaN,32.001734,-96.032076,32.001894,-96.032320,30,4.0,00001
6033,00100008,-96.009801,31.979878,POINT (-96.0098 31.97988),well001.zip,well001s.shp,surface,00100008,NaN,NaN,31.979719,-96.009553,31.979880,-96.009797,15,10.0,00008
8422,00100037,-95.548114,31.557517,POINT (-95.54811 31.55752),well001.zip,well001s.shp,surface,00100037,NaN,NaN,31.557344,-95.547880,31.557520,-95.548110,15,7.0,00037
9699,00100038,-95.994286,32.008876,POINT (-95.99429 32.00888),well001.zip,well001s.shp,surface,00100038,NaN,NaN,32.008718,-95.994038,32.008878,-95.994281,15,7.0,00038
9698,00100039,-95.991410,32.008862,POINT (-95.99141 32.00886),well001.zip,well001s.shp,surface,00100039,NaN,NaN,32.008704,-95.991162,32.008864,-95.991405,15,7.0,00039


### 4.9 Merge Well/Lease Records With Their Coordinates

This is the `api8` join described in Section 0: every well in `lease_wells` (from
`well_api_lease.parquet`) picks up its coordinate (if RRC's shapefiles have one) from
`well_coords_one`. The result, `lease_well_coordinates`, is the master well-level file: one row per
well, carrying both its `lease_key` (for joining to production later) and its coordinates.

In [31]:
lease_wells_geo = lease_wells.merge(
    well_coords_one,
    on="api8",
    how="left",
    suffixes=("", "_gis")
)

lease_wells_geo = gpd.GeoDataFrame(
    lease_wells_geo,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Lease/well rows:", lease_wells.shape)
print("Lease/well rows with coordinates:", lease_wells_geo["geometry"].notna().sum())
print(f"Coordinate match rate: {lease_wells_geo['geometry'].notna().mean():.2%}")


Lease/well rows: (587614, 16)
Lease/well rows with coordinates: 584135
Coordinate match rate: 99.41%


A quick look at wells that *didn't* get a coordinate match — useful for diagnosing whether missing
coordinates cluster in particular counties, districts, or API ranges.

In [32]:
unmatched_lease_wells = lease_wells_geo[lease_wells_geo["geometry"].isna()].copy()

print("Unmatched well/lease rows:", unmatched_lease_wells.shape)

unmatched_lease_wells[
    [
        "api_county_code",
        "api_unique_no",
        "api_no",
        "api8",
        "county_name",
        "oil_gas_code",
        "district_no",
        "lease_no"
    ]
].head(30)


Unmatched well/lease rows: (3479, 32)


,api_county_code,api_unique_no,api_no,api8,county_name,oil_gas_code,district_no,lease_no
361,325,00706,32500706,32500706,MEDINA,O,01,01718
421,249,02759,24902759,24902759,JIM WELLS,O,04,11296
423,249,03067,24903067,24903067,JIM WELLS,O,04,11296
530,049,80350,04980350,04980350,BROWN,O,08,00091
604,325,80186,32580186,32580186,MEDINA,O,01,01720
614,325,00939,32500939,32500939,MEDINA,O,01,01722
615,325,01017,32501017,32501017,MEDINA,O,01,01722
827,049,80446,04980446,04980446,BROWN,O,08,00191
828,049,80447,04980447,04980447,BROWN,O,08,00191
889,325,80063,32580063,32580063,MEDINA,O,01,00298


### 4.10 Save the Well-Level Coordinate File

`lease_well_coordinates.parquet`/`.geoparquet` is the master well-level coordinate reference — later
pipeline stages (e.g. the Permian Basin subsetting notebook) depend on this exact file.

In [33]:
lease_wells_geo.drop(columns="geometry").to_parquet(
    output_folder / "lease_well_coordinates.parquet",
    index=False
)

lease_wells_geo.to_parquet(
    output_folder / "lease_well_coordinates.geoparquet"
)

log.info("✓ Saved lease_well_coordinates.parquet and .geoparquet")


09:45:52 [INFO] ✓ Saved lease_well_coordinates.parquet and .geoparquet


### 4.11 Count Wells per Lease — No Centroid Computed

This step intentionally stops short of computing a lease centroid (see Section 0 for why). A naive
approach here would project to Texas Centric Albers, union all well points per lease, and take the
centroid — **we skip that entirely**: we don't want a single averaged point per lease, we want to
keep every well's own coordinate.

The only thing we still need from this step is **how many wells with known coordinates exist on each
lease** (`n_wells_with_coordinates`), which is what Part B (Section 5) will divide production by.
This is a much cheaper computation than a centroid — just a count, no geometry math — since we don't
need any projected/unioned geometry at all.

> **Downstream note:** `wells_per_lease.parquet` here carries `lease_key` and
> `n_wells_with_coordinates` only — no centroid coordinates. Any downstream notebook that expects
> `lease_latitude`/`lease_longitude` from this file (e.g. the Permian Basin subsetting notebook, as
> it currently stands) will need a small update to work with this schema — flagging it here so it
> isn't a silent surprise.

In [34]:
lease_points = lease_wells_geo[
    lease_wells_geo["geometry"].notna()
].copy()

lease_points = gpd.GeoDataFrame(
    lease_points,
    geometry="geometry",
    crs=lease_wells_geo.crs if lease_wells_geo.crs is not None else "EPSG:4326"
)

print("Rows with geometry:", len(lease_points))
print("Unique leases with geometry:", lease_points["lease_key"].nunique())


Rows with geometry: 584135
Unique leases with geometry: 169051


In [35]:
lease_well_counts = (
    lease_points
    .groupby("lease_key")
    .agg(n_wells_with_coordinates=("api8", "nunique"))
    .reset_index()
)

lease_well_counts.head()


,lease_key,n_wells_with_coordinates
0,O_01_00002,4
1,O_01_00003,2
2,O_01_00005,2
3,O_01_00006,1
4,O_01_00013,1


In [36]:
lease_well_counts.to_parquet(
    output_folder / "wells_per_lease.parquet",
    index=False
)

log.info("✓ Saved wells_per_lease.parquet (lease_key, n_wells_with_coordinates)")


09:45:52 [INFO] ✓ Saved wells_per_lease.parquet (lease_key, n_wells_with_coordinates)


## 5. Part B — Distributing Lease Production Across Wells

### 5.1 Filter and Key the Disposition Data

Restrict to production dated after `POST_DATE_FILTER` (2011-01-01), since well-count and coordinate
coverage are considered more reliable in recent years, and build `lease_key` on `prod_disp` using the
**exact same** normalization functions defined in Section 4.2 — not redefined, just reused, so the
two files' keys are guaranteed to be formatted identically. The raw key columns (`oil_gas_code`,
`district_no`, `lease_no`, and their `_norm` intermediates) are dropped afterward since `lease_key`
alone is what we join on going forward.

In [37]:
prod_disp.date.tail(5)


16761944   1993-02-01
16761945   1993-03-01
16761946   1993-04-01
16761947   1993-05-01
16761948   1993-06-01
Name: date, dtype: datetime64[us]

In [38]:
prod_disp = prod_disp.loc[prod_disp['date'] > POST_DATE_FILTER].copy()
print("Rows after date filter:", prod_disp.shape)


Rows after date filter: (7534532, 29)


In [39]:
prod_disp = add_lease_key(prod_disp)   # same function defined in Section 4.2

prod_disp[["oil_gas_code", "district_no", "lease_no", "lease_key"]].head()


,oil_gas_code,district_no,lease_no,lease_key
0,O,08,09073,O_08_09073
1,O,10,27510,O_10_27510
2,O,10,27510,O_10_27510
3,O,10,27510,O_10_27510
4,O,10,27541,O_10_27541


In [40]:
prod_disp = prod_disp.drop(columns=[
    'oil_gas_code', 'district_no', 'lease_no',
    'oil_gas_code_norm', 'district_no_norm', 'lease_no_norm',
])

prod_disp.columns.tolist()


['field_no',
 'operator_no',
 'operator_name',
 'oil_pipeline_bbl',
 'oil_truck_bbl',
 'oil_tankcar_bbl',
 'oil_tank_cleaning_bbl',
 'oil_circulating_bbl',
 'oil_lost_stolen_bbl',
 'oil_bsw_repressure_bbl',
 'oil_legacy_bbl',
 'oil_skimmed_bbl',
 'oil_scrubber_bbl',
 'oil_no_disp_code_bbl',
 'csgd_field_ops_fuel_mcf',
 'csgd_transmission_mcf',
 'csgd_processing_plant_mcf',
 'csgd_vented_flared_mcf',
 'csgd_gas_lift_mcf',
 'csgd_repressure_mcf',
 'csgd_carbon_black_mcf',
 'csgd_underground_storage_mcf',
 'csgd_no_disp_code_mcf',
 'oil_sold_total_bbl',
 'total_vented_flared_mcf',
 'date',
 'lease_key']

### 5.2 Why This Step Is Chunked

The next operation — joining `prod_disp` (potentially several million lease-month rows spanning
2011–present) onto `lease_well_coordinates` (one row per well) on `lease_key` — is exactly the
row-explosion join described in Section 0: every lease-month row is duplicated once per well on that
lease. Doing this for the *entire* multi-year history in one pass, while also carrying a `geometry`
column for every row, is what has been known to crash the kernel on typical laptop-scale memory
(~16 GB RAM).

To make this robust, Section 5.3 processes `prod_disp` **one calendar year at a time**:

1. Slice out a single year of `prod_disp`.
2. Merge that slice against `lease_well_coordinates` (kept resident in memory the whole time — at
   ~500k rows it's small) and against `lease_well_counts` (also small) to attach coordinates and
   `n_wells_with_coordinates`.
3. Divide the production/disposition columns by `n_wells_with_coordinates`.
4. Write the year's result to its own checkpoint file in `checkpoint_folder`.
5. Free the year's intermediate frames from memory before moving to the next year.

**Crash recovery:** if the kernel dies partway through, just re-run the Section 5.3 cell. It checks
for an existing checkpoint file before processing each year and skips any year that's already done,
so you only redo the year that was in progress when it crashed — not the whole history. If you want
to force a full re-run (e.g. after changing the division logic), delete the `checkpoint_folder`
directory first.

In [41]:
prod_cols = [
    'oil_pipeline_bbl', 'oil_truck_bbl', 'oil_tankcar_bbl', 'oil_tank_cleaning_bbl',
    'oil_circulating_bbl', 'oil_lost_stolen_bbl', 'oil_bsw_repressure_bbl',
    'oil_legacy_bbl', 'oil_skimmed_bbl', 'oil_scrubber_bbl',
    'oil_no_disp_code_bbl', 'csgd_field_ops_fuel_mcf',
    'csgd_transmission_mcf', 'csgd_processing_plant_mcf',
    'csgd_vented_flared_mcf', 'csgd_gas_lift_mcf', 'csgd_repressure_mcf',
    'csgd_carbon_black_mcf', 'csgd_underground_storage_mcf',
    'csgd_no_disp_code_mcf', 'oil_sold_total_bbl',
    'total_vented_flared_mcf',
]
prod_cols = [c for c in prod_cols if c in prod_disp.columns]

well_coord_cols = ["lease_key", "api8", "well_no", "county_name", "longitude", "latitude", "geometry"]
well_coord_cols = [c for c in well_coord_cols if c in lease_wells_geo.columns]

years = sorted(prod_disp["date"].dt.year.unique())
print(f"Processing {len(years)} year(s) in chunks: {years}")


Processing 15 year(s) in chunks: [np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]


### 5.3 Chunked Merge + Equal-Split Division

For each year: merge onto well coordinates (row explosion happens here, scoped to just this year),
merge on `n_wells_with_coordinates`, divide every disposition column by it, and checkpoint to disk.
`n_wells_with_coordinates` is kept in the final output (not dropped) so the equal-split assumption
stays visible in the final data.

In [42]:
for year in years:
    checkpoint_path = checkpoint_folder / f"prod_per_well_approx_{year}.geoparquet"

    if checkpoint_path.exists():
        log.info("Year %s already checkpointed — skipping (%s)", year, checkpoint_path.name)
        continue

    log.info("Processing year %s ...", year)

    year_slice = prod_disp[prod_disp["date"].dt.year == year].copy()

    # Row-explosion join: one lease-month row -> one row per well on that lease
    year_points = year_slice.merge(
        lease_wells_geo[well_coord_cols],
        on="lease_key",
        how="left"
    )
    year_points = gpd.GeoDataFrame(year_points, geometry="geometry", crs="EPSG:4326")

    # Attach n_wells_with_coordinates and divide the lease-level volumes equally across wells
    year_points = year_points.merge(lease_well_counts, on="lease_key", how="left")
    year_points[prod_cols] = year_points[prod_cols].div(year_points["n_wells_with_coordinates"], axis=0)

    match_rate = year_points["geometry"].notna().mean()
    log.info(
        "  year %s: %s lease-month rows -> %s well-month rows (coordinate match rate %.2f%%)",
        year, f"{len(year_slice):,}", f"{len(year_points):,}", match_rate * 100
    )

    year_points.to_parquet(checkpoint_path)

    del year_slice, year_points
    gc.collect()

log.info("✓ All years processed (or already checkpointed).")


09:45:58 [INFO] Processing year 2012 ...
09:45:59 [INFO]   year 2012: 444,034 lease-month rows -> 3,336,940 well-month rows (coordinate match rate 99.50%)
09:46:01 [INFO] Processing year 2013 ...
09:46:02 [INFO]   year 2013: 510,622 lease-month rows -> 3,727,852 well-month rows (coordinate match rate 99.51%)
09:46:05 [INFO] Processing year 2014 ...
09:46:06 [INFO]   year 2014: 533,798 lease-month rows -> 3,786,501 well-month rows (coordinate match rate 99.53%)
09:46:08 [INFO] Processing year 2015 ...
09:46:09 [INFO]   year 2015: 535,436 lease-month rows -> 3,756,615 well-month rows (coordinate match rate 99.55%)
09:46:11 [INFO] Processing year 2016 ...
09:46:12 [INFO]   year 2016: 520,554 lease-month rows -> 3,691,673 well-month rows (coordinate match rate 99.57%)
09:46:14 [INFO] Processing year 2017 ...
09:46:15 [INFO]   year 2017: 518,456 lease-month rows -> 3,670,414 well-month rows (coordinate match rate 99.58%)
09:46:17 [INFO] Processing year 2018 ...
09:46:18 [INFO]   year 2018: 

### 5.4 Concatenate Checkpoints Into the Final Output

Reads every per-year checkpoint back in, concatenates them into a single dataframe, and saves the
final `prod_per_well_approx.parquet` (flat, no geometry) and `.geoparquet` (with geometry), which
include the `n_wells_with_coordinates` column.

The per-year checkpoint files in `checkpoint_folder` are left on disk after this step (not deleted)
in case you need to inspect a single year or re-run the concatenation — feel free to delete that
folder once you've confirmed the final combined file looks correct.

In [43]:
checkpoint_files = sorted(checkpoint_folder.glob("prod_per_well_approx_*.geoparquet"))
print(f"Found {len(checkpoint_files)} checkpoint file(s) to combine.")

year_frames = [gpd.read_parquet(p) for p in checkpoint_files]
prod_well_points = pd.concat(year_frames, ignore_index=True)
prod_well_points = gpd.GeoDataFrame(prod_well_points, geometry="geometry", crs="EPSG:4326")

del year_frames
gc.collect()

print("Combined well-level rows:", len(prod_well_points))
print(f"Rows with coordinates: {prod_well_points['geometry'].notna().mean():.2%}")

prod_well_points[
    ["date", "oil_sold_total_bbl", "lease_key", "latitude", "longitude", "n_wells_with_coordinates"]
].head(10)


Found 15 checkpoint file(s) to combine.
Combined well-level rows: 51189274
Rows with coordinates: 99.59%


,date,oil_sold_total_bbl,lease_key,latitude,longitude,n_wells_with_coordinates
0,2012-03-01,141.0,O_01_13502,30.579891,-96.925623,1.0
1,2012-04-01,188.0,O_01_13502,30.579891,-96.925623,1.0
2,2012-05-01,137.0,O_01_13502,30.579891,-96.925623,1.0
3,2012-06-01,216.0,O_01_13502,30.579891,-96.925623,1.0
4,2012-07-01,134.0,O_01_13502,30.579891,-96.925623,1.0
5,2012-08-01,130.0,O_01_13502,30.579891,-96.925623,1.0
6,2012-09-01,152.0,O_01_13502,30.579891,-96.925623,1.0
7,2012-10-01,154.0,O_01_13502,30.579891,-96.925623,1.0
8,2012-11-01,107.0,O_01_13502,30.579891,-96.925623,1.0
9,2012-12-01,140.0,O_01_13502,30.579891,-96.925623,1.0


In [44]:
prod_well_points.drop(columns="geometry").to_parquet(
    output_folder / "prod_per_well_approx.parquet",
    index=False
)

prod_well_points.to_parquet(
    output_folder / "prod_per_well_approx.geoparquet"
)

log.info("✓ Saved prod_per_well_approx.parquet and .geoparquet (with n_wells_with_coordinates retained)")


09:49:40 [INFO] ✓ Saved prod_per_well_approx.parquet and .geoparquet (with n_wells_with_coordinates retained)


## 6. Quick Visual Sanity Check

Plots a random sample of well points on an interactive map, colored/labeled by
`n_wells_with_coordinates` so you can visually confirm that wells on the same lease cluster together
and share the same split factor.

In [45]:
sample_n = min(5000, prod_well_points["geometry"].notna().sum())

prod_well_points.dropna(subset=["geometry"]).sample(
    sample_n,
    random_state=42
).explore(
    tiles="CartoDB positron",
    tooltip=[
        "lease_key",
        "county_name",
        "oil_sold_total_bbl",
        "n_wells_with_coordinates",
    ]
)


## 7. Summary

This notebook produced three files in `output_folder`:

| File | Grain | Description |
|---|---|---|
| `lease_well_coordinates.parquet` / `.geoparquet` | one row per well | Every well with its `lease_key`, `api8`, and coordinates |
| `wells_per_lease.parquet` | one row per lease | `lease_key` + `n_wells_with_coordinates` (no centroid columns — see Section 4.11 note) |
| `prod_per_well_approx.parquet` / `.geoparquet` | one row per well per month | Lease-level disposition data equally split across wells, with `n_wells_with_coordinates` retained for transparency |
